# E5 GRPO bonus — continue from the completed E0–E4 output

This continuation restores the saved E3 step-300 checkpoint and the exact dataset/evidence from Version 8, then runs only E5.

In [ ]:
# Continue from the completed E0–E4 Kaggle output; run only the E5 bonus.
import base64, gc, importlib.metadata, io, json, os, pathlib, shutil, subprocess, sys, zipfile
from pathlib import Path

# Restrict this GRPO run to one T4 to avoid multi-GPU NF4 model sharding.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
print("GPU:", torch.cuda.get_device_name(0))
assert "T4" in torch.cuda.get_device_name(0), "E5 is configured for a T4"

# Kaggle mounts the previous version's output as an input after it is added in the Input panel.
input_roots = [p for p in Path("/kaggle/input").rglob("labrepo")
               if (p / "student" / "e5.py").is_file() and (p / "data" / "splits" / "train.jsonl").is_file()]
assert input_roots, "Add Version 8 notebook output as Input before Save Version → Run All"
INPUT_ROOT = input_roots[0]
print("Prior run output:", INPUT_ROOT)

ROOT = Path("/kaggle/working/labrepo")
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
for name in ("scripts", "student", "student_tests", "configs"):
    shutil.copytree(INPUT_ROOT / name, ROOT / name)
for name in ("requirements-colab.txt", "pytest.ini"):
    source = INPUT_ROOT / name
    if source.is_file(): shutil.copy2(source, ROOT / name)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as zf:
    zf.extractall(ROOT)
# Keep the full dataset read-only in the attached Kaggle output.
(ROOT / "data").symlink_to(INPUT_ROOT / "data", target_is_directory=True)
(ROOT / "work").mkdir()
for name in ("e3_test.jsonl", "e3_run.json"):
    source = INPUT_ROOT / "work" / name
    if source.is_file(): shutil.copy2(source, ROOT / "work" / name)
(ROOT / "report" / "evidence").mkdir(parents=True)
source_e3 = INPUT_ROOT / "report" / "evidence" / "e3.json"
assert source_e3.is_file(), "Prior run output is missing the real E3 test evidence"
shutil.copy2(source_e3, ROOT / "report" / "evidence" / "e3.json")

# Restore only the selected E3 300-step SFT checkpoint, not the old scratch workspace.
checkpoint_roots = list(Path("/kaggle/input").rglob("day10_clean_v2"))
source_exp = next((p / "e3" for p in checkpoint_roots
                   if (p / "e3" / "step-0300" / "adapter").is_dir()), None)
assert source_exp is not None, "Version 8 output is missing the E3 step-0300 adapter"
CHECKPOINT_ROOT = Path("/kaggle/working/day10_clean_v2")
target_exp = CHECKPOINT_ROOT / "e3"
target_exp.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(source_exp, target_exp)
latest = target_exp / "latest.txt"
latest.write_text(str(target_exp / "step-0300"), encoding="utf-8")
assert (target_exp / "step-0300" / "adapter").is_dir()
print("Restored checkpoint:", target_exp / "step-0300")

os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
# Keep GRPO trainable LoRA leaves in fp32 and disable GradScaler on T4.
e5_path = ROOT / "student" / "e5.py"
e5 = e5_path.read_text(encoding="utf-8")
if "param.data = param.data.float()" not in e5:
    marker = "        param.requires_grad_(train_language_lora)"
    assert e5.count(marker) == 1
    e5 = e5.replace(marker, marker + "\n        if train_language_lora and param.dtype != torch.float32:\n            param.data = param.data.float()", 1)
e5 = e5.replace("fp16=True, bf16=False", "fp16=False, bf16=False")
assert "param.data = param.data.float()" in e5 and "fp16=False, bf16=False" in e5
e5_path.write_text(e5, encoding="utf-8")
os.environ["PYTHONPATH"] = str(ROOT)
BASE_EXP = "e3"
print("E5 continuation is ready; previous E0–E4 artifacts and checkpoint are reused.")

## Reward tests and smoke gates

In [ ]:
# Reward unit checks and the required GRPO smoke gates on the same restored checkpoint.
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pytest", "-q", "student_tests/test_student_e5.py"], check=True)
for steps, limit in ((1, 4), (5, 20)):
    subprocess.run([sys.executable, "-m", "student.e5", "--mode", "train",
                    "--base-exp", BASE_EXP, "--smoke", "--max-steps", str(steps),
                    "--train-limit", str(limit), "--checkpoint-root", str(CHECKPOINT_ROOT),
                    "--output-root", f"/kaggle/working/e5_smoke_{steps}", "--no-resume"], check=True)
print("E5 reward tests and both smoke gates passed.")

## Full 60-step GRPO and test evaluation

In [ ]:
# Full bonus: 60 GRPO steps, test inference, reward-hacking analysis.
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "student.e5", "--mode", "train",
                "--base-exp", BASE_EXP, "--max-steps", "60",
                "--checkpoint-root", str(CHECKPOINT_ROOT)], check=True)
subprocess.run([sys.executable, "-m", "student.e5", "--mode", "infer",
                "--split", "test", "--checkpoint-root", str(CHECKPOINT_ROOT)], check=True)
subprocess.run([sys.executable, "-m", "student.e5", "--mode", "analyze",
                "--baseline-exp", BASE_EXP], check=True)

## Verify and package evidence

In [ ]:
from pathlib import Path
import json, shutil
for name in ("e5_train_log.json", "e5_completion_samples.jsonl", "e5.json", "e5_analysis.json"):
    p = ROOT / "report" / "evidence" / name
    assert p.is_file(), f"Missing E5 evidence: {p}"
    print(name, p.stat().st_size, "bytes")
print((ROOT / "report" / "evidence" / "e5_analysis.json").read_text(encoding="utf-8"))
shutil.make_archive("/kaggle/working/e5_bonus_evidence", "zip", ROOT / "report" / "evidence")
print("Created /kaggle/working/e5_bonus_evidence.zip")